In [ ]:
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from zipfile import ZipFile
from datetime import datetime

device = "cuda" if torch.cuda.is_available() else "cpu"

# ===== アンサンブルしたいrun_dirを並べる =====
model_dirs = [
    Path("outputs/20260608_xxxx_eegnet_subject_seed1234"),
    Path("outputs/20260608_xxxx_eegnet_subject_seed42"),
    Path("outputs/20260608_xxxx_eegnet_subject_seed2026"),
]

# ===== test loader =====
test_set = ThingsEEGDataset("test")
test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=512,
    shuffle=False
)

def predict_proba(model_path):
    model = EEGNetClassifier(
        num_classes=5,
        num_channels=test_set.num_channels,
        seq_len=test_set.seq_len,
        F1=32,
        D=2,
        F2=64,
        dropout=0.5,
        subject_emb_dim=16,
        num_subjects=10,
    ).to(device)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    probs = []

    with torch.no_grad():
        for X, subject_idxs in tqdm(test_loader, leave=False):
            X = X.to(device)
            subject_idxs = subject_idxs.to(device) - 1

            logits = model(X, subject_idxs)
            p = F.softmax(logits, dim=1)

            probs.append(p.cpu().numpy())

    return np.concatenate(probs, axis=0)

# ===== 各モデルのsoftmax確率を取得 =====
all_probs = []

for model_dir in model_dirs:
    model_path = model_dir / "model_best.pt"
    print("Loading:", model_path)
    probs = predict_proba(model_path)
    print(probs.shape)
    all_probs.append(probs)

# ===== softmax平均 =====
avg_probs = np.mean(all_probs, axis=0)

# ===== 最終予測 =====
preds = avg_probs.argmax(axis=1)
print("preds shape:", preds.shape)
print("pred label count:", np.bincount(preds, minlength=5))

# ===== 保存 =====
ensemble_name = "ensemble_eegnet_subject"
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
out_dir = Path("outputs") / f"{timestamp}_{ensemble_name}"
out_dir.mkdir(parents=True, exist_ok=True)

np.save(out_dir / "submission.npy", preds)

print(f"Saved: {out_dir / 'submission.npy'}")

In [ ]:
notebook_path = Path("notebooks") / "ensemble.ipynb"
zip_name = out_dir / f"{timestamp}_{ensemble_name}_submission.zip"

# notebook名は自分の実ファイル名に合わせて変更
with ZipFile(zip_name, "w") as zf:
    zf.write(out_dir / "submission.npy", arcname="submission.npy")
    zf.write(model_dirs[0] / "model_best.pt", arcname="model_best.pt")
    zf.write(notebook_path, arcname="ensemble.ipynb")

print(f"Created: {zip_name}")

with ZipFile(zip_name, "r") as zf:
    print(zf.namelist())